# Xarray-spatial
### User Guide: Balanced Service Area Partitioning
-----

The `balanced_allocation` function partitions a cost surface into territories of roughly equal **cost-weighted area**. Standard `allocation` assigns each cell to the nearest source by cost distance, which can produce lopsided territories when friction varies across the landscape. `balanced_allocation` adds an iterative bias-adjustment step so that each source ends up responsible for a similar total workload.

**Contents:**
- [Setup](#Setup)
- [1. Standard allocation vs balanced allocation](#1.-Standard-allocation-vs-balanced-allocation)
- [2. Asymmetric friction](#2.-Asymmetric-friction)
- [3. Three sources](#3.-Three-sources)
- [4. Tuning tolerance and iterations](#4.-Tuning-tolerance-and-iterations)
- [5. Barriers](#5.-Barriers)

-----

## Setup

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from xrspatial import cost_distance, allocation
from xrspatial.balanced_allocation import balanced_allocation

In [ ]:
def make_raster(data, res=1.0):
    """Helper: create a DataArray with y/x coordinates."""
    h, w = data.shape
    raster = xr.DataArray(
        data.astype(np.float64),
        dims=['y', 'x'],
        attrs={'res': (res, res)},
    )
    raster['y'] = np.arange(h) * res
    raster['x'] = np.arange(w) * res
    return raster


def plot_territories(alloc_arr, title, source_locs=None, ax=None):
    """Plot a territory allocation with distinct colours per source."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    data = alloc_arr.values if hasattr(alloc_arr, 'values') else alloc_arr
    ids = np.unique(data[np.isfinite(data)])
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(ids), 3)))
    cmap = ListedColormap(colors[:len(ids)])
    im = ax.imshow(data, cmap=cmap, origin='upper', interpolation='nearest')
    ax.set_title(title)
    if source_locs is not None:
        for r, c in source_locs:
            ax.plot(c, r, 'k*', markersize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
    return ax


def territory_stats(alloc_arr, friction_arr):
    """Print cell count and cost-weighted area per territory."""
    alloc = alloc_arr.values if hasattr(alloc_arr, 'values') else alloc_arr
    fric = friction_arr.values if hasattr(friction_arr, 'values') else friction_arr
    ids = sorted(np.unique(alloc[np.isfinite(alloc)]))
    print(f"{'Source':>8}  {'Cells':>6}  {'Cost-weighted area':>18}")
    print('-' * 38)
    for sid in ids:
        mask = alloc == sid
        n = int(np.sum(mask))
        w = float(np.sum(fric[mask]))
        print(f"{sid:>8.0f}  {n:>6d}  {w:>18.1f}")

## 1. Standard allocation vs balanced allocation

With uniform friction, standard `allocation` assigns each cell to the nearest source by cost distance. The resulting territories depend on source placement and have no guarantee of equal size. `balanced_allocation` adjusts boundaries so each territory has roughly the same total friction (cost-weighted area).

In [ ]:
# Two sources placed asymmetrically on a 30x30 grid
source_data = np.zeros((30, 30))
source_data[5, 5] = 1.0    # source near top-left corner
source_data[15, 15] = 2.0  # source near centre

raster = make_raster(source_data)
friction = make_raster(np.ones((30, 30)))
locs = [(5, 5), (15, 15)]

# Standard allocation (nearest by cost)
std_alloc = allocation(raster)

# Balanced allocation
bal_alloc = balanced_allocation(raster, friction, tolerance=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_territories(std_alloc, 'Standard allocation', locs, ax=axes[0])
plot_territories(bal_alloc, 'Balanced allocation', locs, ax=axes[1])
plt.tight_layout()
plt.show()

print('\nStandard allocation:')
territory_stats(std_alloc, friction)
print('\nBalanced allocation:')
territory_stats(bal_alloc, friction)

The balanced version shifts the boundary so both territories end up with a similar number of cells (and since friction is uniform, similar cost-weighted area).

## 2. Asymmetric friction

When friction varies, balancing by cost-weighted area means the territory with expensive cells gets fewer of them, while the territory with cheap cells gets more. The total friction within each territory converges to the same value.

In [ ]:
# Left half has low friction, right half has high friction
source_data2 = np.zeros((30, 30))
source_data2[15, 5] = 1.0   # source on the cheap side
source_data2[15, 25] = 2.0  # source on the expensive side

fric_data2 = np.ones((30, 30))
fric_data2[:, 15:] = 4.0  # right half is 4x more costly

raster2 = make_raster(source_data2)
friction2 = make_raster(fric_data2)
locs2 = [(15, 5), (15, 25)]

bal2 = balanced_allocation(raster2, friction2, tolerance=0.05)

# Also compute cost_distance for reference
cd2 = cost_distance(raster2, friction2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(fric_data2, cmap='YlOrRd', origin='upper')
axes[0].set_title('Friction surface')
plt.colorbar(axes[0].images[0], ax=axes[0], shrink=0.8)

axes[1].imshow(cd2.values, cmap='magma', origin='upper')
axes[1].set_title('Cost distance')
plt.colorbar(axes[1].images[0], ax=axes[1], shrink=0.8)

plot_territories(bal2, 'Balanced allocation', locs2, ax=axes[2])
plt.tight_layout()
plt.show()

print('\nBalanced allocation territory stats:')
territory_stats(bal2, friction2)

Source 1 (cheap side) covers more cells but the same total friction as source 2 (expensive side), which covers fewer cells.

## 3. Three sources

The algorithm works with any number of sources. Each territory converges to roughly 1/N of the total cost-weighted area.

In [ ]:
# Three sources on a grid with some friction variation
np.random.seed(42)
source_data3 = np.zeros((40, 40))
source_data3[8, 20] = 1.0
source_data3[30, 8] = 2.0
source_data3[30, 32] = 3.0

# Smooth random friction field
from scipy.ndimage import gaussian_filter
raw_fric = np.random.uniform(1.0, 5.0, (40, 40))
fric_data3 = gaussian_filter(raw_fric, sigma=3)
fric_data3 = np.clip(fric_data3, 1.0, None)

raster3 = make_raster(source_data3)
friction3 = make_raster(fric_data3)
locs3 = [(8, 20), (30, 8), (30, 32)]

bal3 = balanced_allocation(raster3, friction3, tolerance=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(fric_data3, cmap='YlOrRd', origin='upper')
axes[0].set_title('Friction surface')
for r, c in locs3:
    axes[0].plot(c, r, 'k*', markersize=12)
plt.colorbar(axes[0].images[0], ax=axes[0], shrink=0.8)

plot_territories(bal3, 'Balanced allocation (3 sources)', locs3, ax=axes[1])
plt.tight_layout()
plt.show()

print('\nTerritory stats:')
territory_stats(bal3, friction3)

## 4. Tuning tolerance and iterations

The `tolerance` parameter controls how close to equal the territories need to be (as a fraction of the mean). Tighter tolerance means more iterations. The `learning_rate` controls how aggressively biases are updated each step.

In [ ]:
# Compare different tolerance values
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, tol in zip(axes, [0.20, 0.05, 0.01]):
    result = balanced_allocation(
        raster3, friction3, tolerance=tol, max_iterations=200,
    )
    plot_territories(result, f'tolerance={tol}', locs3, ax=ax)
    
    # Compute balance quality
    vals = result.values
    weights = []
    for sid in [1.0, 2.0, 3.0]:
        weights.append(float(np.sum(fric_data3[vals == sid])))
    mean_w = np.mean(weights)
    max_dev = max(abs(w - mean_w) / mean_w for w in weights)
    ax.set_xlabel(f'max deviation: {max_dev:.1%}')

plt.tight_layout()
plt.show()

## 5. Barriers

NaN cells in the friction surface act as impassable barriers, just like in `cost_distance`. The balanced allocation respects these barriers and leaves unreachable cells as NaN.

In [ ]:
# Grid with a diagonal barrier
source_data4 = np.zeros((30, 30))
source_data4[5, 5] = 1.0
source_data4[25, 25] = 2.0

fric_data4 = np.ones((30, 30))
# Diagonal barrier with a gap
for i in range(30):
    if abs(i - 15) > 2:  # gap near the middle
        fric_data4[i, i] = np.nan
        if i + 1 < 30:
            fric_data4[i, i + 1] = np.nan

raster4 = make_raster(source_data4)
friction4 = make_raster(fric_data4)
locs4 = [(5, 5), (25, 25)]

bal4 = balanced_allocation(raster4, friction4, tolerance=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

barrier_vis = fric_data4.copy()
barrier_vis[np.isnan(barrier_vis)] = 0
axes[0].imshow(barrier_vis, cmap='gray', origin='upper')
axes[0].set_title('Friction (dark = barrier)')
for r, c in locs4:
    axes[0].plot(c, r, 'r*', markersize=12)

plot_territories(bal4, 'Balanced allocation with barrier', locs4, ax=axes[1])
plt.tight_layout()
plt.show()

print('\nTerritory stats:')
territory_stats(bal4, friction4)

Both sources must route through the gap in the barrier. The balanced allocation still equalizes the cost-weighted area on each side, subject to the constraint that paths must go through the gap.

-----

## Summary

| Parameter | Default | What it does |
|---|---|---|
| `raster` | (required) | Source raster with non-zero values as source IDs |
| `friction` | (required) | Cost surface (positive values, NaN = impassable) |
| `tolerance` | 0.05 | Convergence threshold (fraction of mean) |
| `max_iterations` | 100 | Maximum balancing iterations |
| `learning_rate` | 0.5 | Bias update aggressiveness |
| `max_cost` | inf | Cost budget passed to `cost_distance` |
| `connectivity` | 8 | 4 or 8 pixel connectivity |